# 07 – Intent Classification & Routing

The **intent module** classifies natural language queries and maps them to the right agents.  
It uses an LLM with structured output, falling back to keyword matching when no API key is set.

This notebook tests the classification and routing logic end-to-end.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
os.environ['ENABLE_MOCK'] = 'true'

In [ ]:
from graph.intent import classify_intent, QueryIntent, KEYWORD_MAP
from graph.routing import get_agents_for_intent, INTENT_AGENT_MAP

## 1. All supported intent types

In [ ]:
print('All intent types:')
for intent in QueryIntent:
    agents = get_agents_for_intent(intent.value)
    print(f'  {intent.value:<22} => agents: {agents}')

## 2. Classify sample queries (keyword fallback)

In [ ]:
sample_queries = [
    'Why did retention drop last month?',
    'What is the DQ score for CAC?',
    'Who owns the bookings dataset?',
    'Open Jira bugs for retention pipeline',
    'What is GRR?',
    'Create a bug ticket for missing LTV data',
    'Update metadata owner for bookings',
    'Create a new DQ rule for completeness',
    'Show me all metrics and governance status',
    'How does the CAC trend look this quarter?',
]

print(f"{'Query':<50} {'Intent':<22} {'Confidence':>10} {'Products'}")
print('-' * 105)
for q in sample_queries:
    c = classify_intent(q)
    print(f"{q[:49]:<50} {c.intent.value:<22} {c.confidence:>10.2f} {c.data_products}")

## 3. Intent → Agent routing

In [ ]:
print('Intent to Agent mapping:')
print(f"{'Intent':<22} {'Agents'}")
print('-' * 60)
for intent, agents in INTENT_AGENT_MAP.items():
    print(f"  {intent:<22} {agents}")

## 4. Keyword map inspection

In [ ]:
print('Keyword triggers per intent:')
for intent, keywords in KEYWORD_MAP.items():
    print(f'  {intent}: {keywords}')

## 5. Test write intents

In [ ]:
write_queries = [
    ('create ticket for bookings pipeline failure', 'write_ticket'),
    ('open ticket: LTV calculation error', 'write_ticket'),
    ('update metadata owner for retention dataset', 'write_metadata'),
    ('create rule: bookings must not be negative', 'write_rule'),
    ('add rule for data quality validation', 'write_rule'),
    ('define rule: GRR threshold check', 'write_rule'),
]

print('Write intent detection:')
for query, expected in write_queries:
    c = classify_intent(query)
    match = '' if c.intent.value == expected else ' MISMATCH'
    print(f"  '{query}'")
    print(f"    expected={expected}, got={c.intent.value}{match}")

## 6. Agent list for each intent

In [ ]:
# full_diagnostic triggers all four read agents
agents = get_agents_for_intent('full_diagnostic')
print(f'full_diagnostic agents: {agents}')
assert 'information' in agents
assert 'knowledge' in agents
assert 'metadata' in agents
assert 'capacity' in agents
print('All expected agents present for full_diagnostic')